# Whale-song space-time reconstruction (combined notebook)

This combines two things your collaborator gave you:

- **`whalesong_recovery.ipynb`** — has the *real* whale audio, but wrongly treats
  it as a function of **fiber position** (distance) rather than time. It recovers
  one single spatial trace, with the whole audio smashed onto the range axis.
- **`OFDR_All_Five_Signals_Trace_Comparison_animations.ipynb`** — has the correct
  **space-time** formalism, g(z, T): a whole spatial trace recovered at every
  interrogation repetition (slow time T). But its whale call is *synthetic*
  (`call_packet`), not real audio.

Here we build a real g(z, T) field from the actual whale recording, run it
through the same OFDR matched-filter recovery as the second notebook, and then
**stitch the recovered spatial traces back together over T** to get the
recovered audio back out — plus an animation of the space-time reconstruction.

**Seismic and temperature channels are omitted** (whale-only, per your request).

### The one assumption we have to make

The audio file only contains *time* information — no spatial information at
all. So we have to assume how the whale's signal couples into the fiber in
space. We assume:

- the whale sits near one point `z0` along the cable,
- its coupling into the fiber falls off as a Gaussian with distance from `z0`
  (a "local disturbance," similar in spirit to the synthetic notebook's
  `whale_coupling`)

In [ ]:
import numpy as np
from scipy.io import wavfile
from scipy.interpolate import interp1d
from scipy.signal import fftconvolve
from scipy.ndimage import uniform_filter1d
from scipy import fft as sfft
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

np.random.seed(0)
plt.rcParams["figure.figsize"] = (10, 5)

## 1. Load the real whale audio

`whalesong_original.wav` here is your mp3 decoded to 8 kHz mono PCM
(`ffmpeg -i whalesong_wav.mp3 -ar 8000 -ac 1 whalesong_original.wav`).
Point `AUDIO_PATH` at your own file if it differs.

In [ ]:
AUDIO_PATH = "whalesong.wav"

audio_sr, audio_raw = wavfile.read(AUDIO_PATH)
audio_raw = audio_raw.astype(np.float64)
audio_norm = audio_raw / np.max(np.abs(audio_raw))       # normalize to [-1, 1]
n_audio = len(audio_norm)
audio_time = np.arange(n_audio) / audio_sr                # seconds

print(f"sample rate = {audio_sr} Hz, n = {n_audio}, duration = {n_audio/audio_sr:.3f} s")
plt.plot(audio_time, audio_norm, linewidth=0.5)
plt.title("Whale audio (time domain) -- what we're actually trying to recover")
plt.xlabel("time (s)")
plt.show()

## 2. Fiber / OFDR parameters

Key idea: pick the fiber length so the round-trip time is **exactly**
`1 / audio_sr`. Then one interrogation pulse (one row of slow time `T`)
corresponds to exactly one audio sample -- no resampling needed anywhere,
and the recovered slow-time axis is already the audio's own time axis.

We also had to retune the chirp rate `gamma` from `whalesong_recovery.ipynb`'s
`5e12`: that value was tuned for a 100 km fiber and badly aliases against the
fast-time sampling on this much shorter fiber (its swept frequency ends up
far above the fast-time Nyquist rate). We instead pick `gamma` so the chirp's
swept bandwidth is a safe fraction of the Nyquist frequency.

In [ ]:
refractive_index = 1.468
speed_of_light_m_per_s = 299_792_458.0
speed_in_fiber_m_per_s = speed_of_light_m_per_s / refractive_index

fiber_length_km = 10
fiber_length_m = fiber_length_km * 1000

num_ranges = 1501                                   # spatial (fast-time) samples along the fiber
tau = np.linspace(0.0, fiber_length_m / speed_in_fiber_m_per_s, num_ranges)
z_km = tau * speed_in_fiber_m_per_s / 1000.0
fast_time = 2.0 * tau
dt_tau = tau[1] - tau[0]
dt_fast = fast_time[1] - fast_time[0]

# --- pulse repetition interval, decoupled from audio_sr ---------------
round_trip_s = 2 * tau[-1]
safety_factor = 1.2                       # >=1; margin so you're not right at the edge
pulse_interval_s = safety_factor * round_trip_s
playback_sr = int(round(1.0 / pulse_interval_s))

n_slow = int(np.floor(audio_time[-1] / pulse_interval_s)) + 1
slow_time = np.arange(n_slow) * pulse_interval_s   # this REPLACES `slow_time = audio_time` later on

max_prr_hz = 1.0 / pulse_interval_s
print(f"round trip     = {round_trip_s*1e6:.3f} us")
print(f"pulse interval = {pulse_interval_s*1e6:.3f} us  (max pulse repetition rate = {max_prr_hz:.1f} Hz)")
if max_prr_hz < audio_sr:
    print(f"WARNING: max PRR ({max_prr_hz:.1f} Hz) < audio_sr ({audio_sr} Hz) "
          f"-- you cannot resolve the full audio bandwidth on this fiber length.")
# -----------------------------------------------------------------------------

f_0 = 0.0
alpha = 3.5e-6
t_pulse_end = 0.3 * tau[-1]

nyquist_fast = 1.0 / (2.0 * dt_fast)
bandwidth_target = 0.8 * nyquist_fast
gamma = bandwidth_target / t_pulse_end

print(f"fiber length   = {fiber_length_km:.4f} km")
print(f"num_ranges     = {num_ranges}")
print(f"gamma          = {gamma:.3e}")

In [ ]:
def generate_chirp(t, t_pulse_end=None):
    c = np.exp(2j * np.pi * (f_0 * t + gamma / 2 * t ** 2))
    if t_pulse_end is not None:
        c = np.where((t >= 0) & (t <= t_pulse_end), c, 0.0)
    return c

n_pulse = int(np.floor(t_pulse_end / dt_fast)) + 1
pulse_time = np.arange(n_pulse) * dt_fast
interrogation = generate_chirp(pulse_time, t_pulse_end)

plt.plot(pulse_time, interrogation.real)
plt.title("Interrogation chirp (real part)")
plt.xlabel("pulse time (seconds)")
plt.show()

## 3. Build the real space-time whale field g(z, T)

The audio is a function of time
(`T`, one-to-one with slow time), and gets a small, explicit **spatial**
model layered on top -- the piece the recording itself can't tell us. The spatial model for the whale call originates in the middle of the cable and propogates outwards. The effect of the whale call decays as the sound travels further from the whale.

In [ ]:
z0_km = z_km[len(z_km) // 2]          # assume the whale is near the middle of the fiber
sigma_z_km = 0.2 * fiber_length_km    # spatial footprint width (controls how far the sound travels before decaying)
target_peak_phase = 0.8               # radians

audio_interp = interp1d(audio_time, audio_norm, bounds_error=False, fill_value=0.0)

spatial_gaussian = np.exp(-0.5 * ((z_km - z0_km) / sigma_z_km) ** 2)
whale_field = target_peak_phase * audio_interp(slow_time[:, None]) * spatial_gaussian[None, :]

z_index = int(np.argmax(spatial_gaussian))   # read-out position = peak of the coupling
print(f"z0 = {z0_km:.3f} km, sigma_z = {sigma_z_km:.3f} km, "
      f"readout z_index = {z_index} (z = {z_km[z_index]:.3f} km)")
print("whale_field shape (n_T, n_z):", whale_field.shape)

Add acoustic-propagation-delay effect

In [ ]:
c_sound_m_per_s = 1500.0
delay_s_per_km = 1000.0 / c_sound_m_per_s
u = slow_time[:, None] - delay_s_per_km * np.abs(z_km[None, :] - z0_km)
whale_field = target_peak_phase * audio_interp(u) * spatial_gaussian[None, :]

## 4. OFDR forward model + matched-filter recovery

In [ ]:
def reflection_profile(rng):
    """
    generate reflection profile of the cable
    """
    random_phase = rng.uniform(0.0, 2 * np.pi, size=num_ranges)
    random_amplitude = rng.normal(size=num_ranges)
    attenuation = np.sqrt(np.exp(-alpha * z_km * 1000.0))
    return random_amplitude * attenuation * np.exp(1j * random_phase)


def matched_output(profile_matrix, interrogation):
    """
    compute matched filter return signal
    """
    profiles = np.asarray(profile_matrix, dtype=complex)
    was_vector = profiles.ndim == 1
    if was_vector:
        profiles = profiles[None, :]

    returned_full = fftconvolve(profiles, interrogation[None, :], mode="full", axes=1) * dt_tau
    returned = returned_full[:, :num_ranges]

    matched_full = fftconvolve(returned, np.conj(interrogation[::-1])[None, :], mode="full", axes=1) * dt_fast
    start = len(interrogation) - 1
    result = matched_full[:, start:start + num_ranges]
    return result[0] if was_vector else result


def recover_field(h_x, h_y, true_field, interrogation, batch=4000):
    """
    From an x and y polarization of light, compute the spatio-temporal recovery of the true_field
    Batched over slow time T to keep memory bounded for long audio clips.
    """
    m_ideal_x = matched_output(h_x, interrogation)
    m_ideal_y = matched_output(h_y, interrogation)

    n_T = true_field.shape[0]
    out = np.empty_like(true_field)
    for start in range(0, n_T, batch):
        stop = min(start + batch, n_T)
        phase_factor = np.exp(1j * true_field[start:stop])
        m_pert_x = matched_output(h_x[None, :] * phase_factor, interrogation)
        m_pert_y = matched_output(h_y[None, :] * phase_factor, interrogation)
        combined = m_pert_x * np.conj(m_ideal_x)[None, :] + m_pert_y * np.conj(m_ideal_y)[None, :]
        out[start:stop] = np.angle(combined)
    return out


fiber_seed = 100
rng_fiber = np.random.default_rng(fiber_seed)
h_x = reflection_profile(rng_fiber)
h_y = reflection_profile(rng_fiber)

recovered_field = recover_field(h_x, h_y, whale_field, interrogation)
print("Recovered field shape:", recovered_field.shape)

## 5. Stitch the spatial reconstructions back into audio

At every time step T we now have a full recovered spatial profile ĝ(z, T).
Reading off the same z position at every T and concatenating gives back a
time series -- this is the "stitching" step.

In [ ]:
true_trace = whale_field[:, z_index] / target_peak_phase
rec_trace = recovered_field[:, z_index] / target_peak_phase

# Low-pass filter to knock down high-frequency matched-filter static
# (same trick as whalesong_recovery.ipynb, cell 11)
freq = sfft.rfft(rec_trace)
cutoff_bin = int(len(freq) * 0.35)
freq[cutoff_bin:] = 0.0

corr = np.corrcoef(true_trace, rec_trace)[0, 1]
print(f"correlation (true vs. recovered): {corr:.6f}")

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
t = np.arange(len(true_trace)) / audio_sr
axes[0].plot(t, true_trace, linewidth=0.5)
axes[0].set_title("True whale waveform at the readout position z*")
axes[1].plot(t, rec_trace, linewidth=0.5, color="tab:orange")
axes[1].set_title("Recovered whale waveform")
axes[1].set_xlabel("time (s)")
fig.tight_layout()
plt.show()

In [ ]:
def to_int16(x):
    x = x / np.max(np.abs(x) + 1e-12)
    return (x * 0.95 * 32767).astype(np.int16)

wavfile.write("recovered_whalesong.wav", audio_sr, to_int16(rec_trace))
wavfile.write("true_whalesong_at_readout.wav", audio_sr, to_int16(true_trace))
print("Saved recovered_whalesong.wav")

# Playback of audio files

Our recovered/true traces live at whatever sample rate the fiber physics dictates (playback_sr = 1/pulse_interval_s), which is almost never a "standard" rate that sound cards accept (e.g. 851 Hz, 8509.1 Hz, etc.). These two functions clean the signal up and resample it to a rate the hardware will actually play, without changing its pitch/speed.

In [ ]:
import sounddevice as sd
import numpy as np
from scipy.signal import resample_poly
from math import gcd

def apply_edge_fade(x, fade_samples=50):
    """Taper the first/last `fade_samples` samples of x to zero.

    Why: resample_poly / FFT-based resampling implicitly treats the array as
    one period of a repeating signal. If x[0] and x[-1] aren't both ~0, that
    "wraparound" has a sharp discontinuity, which shows up as an audible
    click/ding at the start of playback. Fading the edges to exactly zero
    removes that discontinuity. fade_samples is tiny relative to the whole
    clip, so it doesn't audibly change the call itself.

    x            : 1D array, the audio samples to fade (copied, not modified in place)
    fade_samples : how many samples at each end to ramp to zero
    """
    x = x.copy()
    fade_samples = min(fade_samples, len(x) // 2)   # don't fade more than half the clip
    fade_in = np.linspace(0, 1, fade_samples)
    fade_out = np.linspace(1, 0, fade_samples)
    x[:fade_samples] *= fade_in
    x[-fade_samples:] *= fade_out
    return x


def play_resampled(x, native_sr, standard_sr=44100, fade_samples=50):
    """Resample x from its native (fiber-derived) rate to a standard audio
    rate, and play it.

    x            : 1D array, the audio samples at their *native* rate
                   (e.g. true_trace)
    native_sr    : the actual sample rate the data represents -- for us this
                   is `playback_sr` (1/pulse_interval_s), NOT audio_sr, since
                   the fiber's max pulse rate sets the real rate of the
                   recovered trace
    standard_sr  : a sample rate real sound hardware supports (44100 or 48000
                   are safe defaults); this is NOT native_sr -- it's just the
                   rate we convert TO so sd.play() doesn't raise
                   PortAudioError: Invalid sample rate
    fade_samples : passed through to apply_edge_fade (see above)
    """
    x = apply_edge_fade(x, fade_samples)

    # resample_poly needs an exact integer ratio up/down. Reducing by the
    # gcd first (rather than passing e.g. up=44100, down=851 directly) keeps
    # the internal filter small and fast -- same result, cheaper to compute.
    native_sr_int = int(round(native_sr))
    g = gcd(standard_sr, native_sr_int)
    up, down = standard_sr // g, native_sr_int // g

    resampled = resample_poly(x, up, down)   # time-domain polyphase resample:
                                              # up-samples by `up`, filters,
                                              # then down-samples by `down`
    sd.play(resampled, standard_sr)

In [ ]:
play_resampled(true_trace, playback_sr)

In [ ]:
play_resampled(rec_trace, playback_sr)

## 6. Space-time animation

Two views:

1. A static full waterfall (z vs T heatmap) over the whole clip -- true vs. recovered.
2. An animated line plot of the *envelope* vs. z, stepping through T at 30 fps.
   (The raw waveform oscillates far faster than any video frame rate can show
   meaningfully, so we animate a 15 ms envelope instead of the raw samples --
   the full-fidelity audio is in the .wav file above.)

In [ ]:
def max_pool_rows_signed(field, block_size):
    """Downsample the time axis by max-magnitude pooling, keeping sign.

    Plain averaging (e.g. what imshow does when it compresses many rows
    into one pixel row) crushes brief peaks toward zero. This instead keeps
    whichever sample was loudest in each block, so isolated peaks (like the
    whale call) survive downsampling instead of getting blurred away.
    """
    n_T, n_z = field.shape                                  # n_T: number of time samples, n_z: number of range bins

    n_blocks = n_T // block_size                             # how many complete blocks of size `block_size` fit in n_T
                                                              # (integer division -- any leftover rows are dropped below)

    trimmed = field[:n_blocks * block_size]                  # drop the leftover rows that don't fill a full block,
                                                              # so the array reshapes evenly

    reshaped = trimmed.reshape(n_blocks, block_size, n_z)    # split the time axis into (n_blocks, block_size) --
                                                              # i.e. group every `block_size` consecutive time
                                                              # samples together, one group per output row

    idx = np.argmax(np.abs(reshaped), axis=1)                # for each block and each z, find WHICH of the
                                                              # `block_size` samples has the largest magnitude
                                                              # (axis=1 is the within-block time axis) --
                                                              # result shape: (n_blocks, n_z)

    return np.take_along_axis(reshaped, idx[:, None, :], axis=1)[:, 0, :]
    # use those indices to pull out the ACTUAL (signed) value at each
    # peak location, not just its magnitude -- idx[:, None, :] reinserts
    # the size-1 time axis so take_along_axis can index correctly, and
    # [:, 0, :] squeezes that axis back out, giving shape (n_blocks, n_z)

# pick block_size so you land near a target number of displayed rows, e.g. ~500
target_rows = 500
block_size = max(1, len(whale_field) // target_rows)

whale_field_pooled = max_pool_rows_signed(whale_field, block_size)
recovered_field_pooled = max_pool_rows_signed(recovered_field, block_size)

# gauge_bins = 5   # ~ a spatial window in range-bin units; tune this
# recovered_field_smoothed = uniform_filter1d(recovered_field, size=gauge_bins, axis=1, mode="nearest")

vmax = target_peak_phase
extent = [z_km[0], z_km[-1], slow_time[-1], slow_time[0]]

fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharey=True)
axes[0].imshow(whale_field_pooled, aspect="auto", extent=extent, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[0].set_title("True g(z, T)")
axes[0].set_xlabel("cable position z (km)")
axes[0].set_ylabel("time T (s)")

im1 = axes[1].imshow(recovered_field_pooled, aspect="auto", extent=extent, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[1].set_title("Recovered ĝ(z, T)")
axes[1].set_xlabel("cable position z (km)")
fig.colorbar(im1, ax=axes.ravel().tolist(), label="phase (rad)", shrink=0.8)
fig.suptitle("Whale call space-time reconstruction along the fiber")
plt.show()

In [ ]:
fps = 30
window_samples = max(1, int(0.0005 * audio_sr))   # 5 ms smoothing window

# env_true = uniform_filter1d(whale_field, size=window_samples, axis=0, mode="nearest")
env_rec = uniform_filter1d(recovered_field, size=window_samples, axis=0, mode="nearest")

env_true = whale_field
# env_rec = recovered_field

current_sr = 1.0 / pulse_interval_s
frame_step = max(1, int(round(current_sr / fps)))
env_true_f = env_true[::frame_step]
env_rec_f = env_rec[::frame_step]
t_f = slow_time[::frame_step]
n_frames = env_true_f.shape[0]

y_max = 1.05 * max(env_true_f.max(), env_rec_f.max())

fig2, ax = plt.subplots(figsize=(6, 4.5))
true_line, = ax.plot(z_km, env_true_f[0], color="black", linewidth=2.2, label="true (envelope)")
rec_line, = ax.plot(z_km, env_rec_f[0], color="tab:blue", linewidth=1.6, label="recovered (envelope)")
ax.set_xlim(z_km[0], z_km[-1])
# ax.set_ylim(-y_max, y_max)
ax.set_ylim(-1, 1)
ax.set_xlabel("cable position z (km)")
ax.set_ylabel("phase envelope (rad)")
ax.grid(alpha=0.25)
ax.legend(loc="upper right", fontsize=9)
title = ax.set_title(f"T = {t_f[0]:.3f} s")
fig2.tight_layout()


def update(i):
    true_line.set_ydata(env_true_f[i])
    rec_line.set_ydata(env_rec_f[i])
    title.set_text(f"T = {t_f[i]:.3f} s")
    return true_line, rec_line, title

anim = animation.FuncAnimation(fig2, update, frames=n_frames, interval=1000 / fps, blit=False)
plt.close(fig2)
display(HTML(anim.to_jshtml()))

## 7. Combined file: animation + recovered audio muxed together

Saves the animation as an .mp4, then uses `ffmpeg` to attach the recovered
whale audio as its soundtrack.

In [ ]:
anim.save("spatial_reconstruction_silent.mp4", writer=animation.FFMpegWriter(fps=fps, codec="libx264", bitrate=2000))

import subprocess
subprocess.run([
    "ffmpeg", "-y",
    "-i", "spatial_reconstruction_silent.mp4",
    "-i", "recovered_whalesong.wav",
    "-c:v", "copy", "-c:a", "aac", "-shortest",
    "-map", "0:v:0", "-map", "1:a:0",
    "whalesong_spacetime_reconstruction.mp4",
], check=True)
print("Saved whalesong_spacetime_reconstruction.mp4")

## 8. True-only animation + audio (combined file)

Same idea as section 7, but using only the **true** g(z, T) field -- no
recovered/comparison curve -- and the **true** whale trace as the
soundtrack instead of the recovered one. Useful as a "ground truth" reference
video to compare the recovered version against.

Reuses `env_true_f`, `t_f`, `frame_step`, and `fps` from the animation cell
above (section 6), so run this after that cell.

In [ ]:
# --- Single-line animation: true g(z, T) only ---------------------------
fig3, ax3 = plt.subplots(figsize=(6, 4.5))
true_only_line, = ax3.plot(z_km, env_true_f[0], color="black", linewidth=2.2, label="true")
ax3.set_xlim(z_km[0], z_km[-1])
ax3.set_ylim(-1, 1)
ax3.set_xlabel("cable position z (km)")
ax3.set_ylabel("phase (rad)")
ax3.grid(alpha=0.25)
ax3.legend(loc="upper right", fontsize=9)
title3 = ax3.set_title(f"T = {t_f[0]:.3f} s")
fig3.tight_layout()


def update_true_only(i):
    true_only_line.set_ydata(env_true_f[i])
    title3.set_text(f"T = {t_f[i]:.3f} s")
    return true_only_line, title3


anim_true_only = animation.FuncAnimation(
    fig3, update_true_only, frames=n_frames, interval=1000 / fps, blit=False
)
plt.close(fig3)
display(HTML(anim_true_only.to_jshtml()))

In [ ]:
# --- True-song audio track, resampled to a standard rate for muxing ------
# AAC (like most sound hardware) only accepts a fixed set of standard sample
# rates -- playback_sr (the fiber-derived rate, e.g. 851 Hz or 8509 Hz) is
# not one of them, so we reuse the same fade + resample_poly approach from
# the playback cells above, but write the result to a .wav instead of
# calling sd.play().
standard_sr = 44100

true_trace_faded = apply_edge_fade(true_trace, fade_samples=50)

native_sr_int = int(round(playback_sr))
g = gcd(standard_sr, native_sr_int)
up, down = standard_sr // g, native_sr_int // g
true_trace_resampled = resample_poly(true_trace_faded, up, down)

wavfile.write("true_whalesong_resampled.wav", standard_sr, to_int16(true_trace_resampled))
print(f"Saved true_whalesong_resampled.wav at {standard_sr} Hz "
      f"(native rate was {playback_sr} Hz)")

In [ ]:
# --- Save the true-only animation as a silent video, then mux in the true audio ---
anim_true_only.save(
    "true_only_spatial_reconstruction_silent.mp4",
    writer=animation.FFMpegWriter(fps=fps, codec="libx264", bitrate=2000),
)

import subprocess
subprocess.run([
    "ffmpeg", "-y",
    "-i", "true_only_spatial_reconstruction_silent.mp4",
    "-i", "true_whalesong_resampled.wav",
    "-c:v", "copy", "-c:a", "aac", "-shortest",
    "-map", "0:v:0", "-map", "1:a:0",
    "true_whalesong_spacetime_reconstruction.mp4",
], check=True)
print("Saved true_whalesong_spacetime_reconstruction.mp4")

## 9. Recovered-only animation + audio (combined file)

Same idea as section 8, but with the **recovered** ĝ(z, T) field instead of
the true one, and `rec_trace` (the recovered audio, before low-pass) as the
soundtrack. This is the "reconstruction only" counterpart to the ground-truth
video above -- useful for sharing just the recovered result on its own,
without the true curve for comparison.

Reuses `env_rec_f`, `t_f`, `frame_step`, and `fps` from the animation cell in
section 6, so run that cell first.

In [ ]:
# --- Single-line animation: recovered ĝ(z, T) only -----------------------
fig4, ax4 = plt.subplots(figsize=(6, 4.5))
rec_only_line, = ax4.plot(z_km, env_rec_f[0], color="tab:blue", linewidth=1.6, label="recovered")
ax4.set_xlim(z_km[0], z_km[-1])
ax4.set_ylim(-1, 1)
ax4.set_xlabel("cable position z (km)")
ax4.set_ylabel("phase (rad)")
ax4.grid(alpha=0.25)
ax4.legend(loc="upper right", fontsize=9)
title4 = ax4.set_title(f"T = {t_f[0]:.3f} s")
fig4.tight_layout()


def update_rec_only(i):
    rec_only_line.set_ydata(env_rec_f[i])
    title4.set_text(f"T = {t_f[i]:.3f} s")
    return rec_only_line, title4


anim_rec_only = animation.FuncAnimation(
    fig4, update_rec_only, frames=n_frames, interval=1000 / fps, blit=False
)
plt.close(fig4)
display(HTML(anim_rec_only.to_jshtml()))

In [ ]:
# --- Recovered-song audio track, resampled to a standard rate for muxing ---
# Same reasoning as section 8: playback_sr (the fiber-derived rate) isn't a
# standard rate AAC/most hardware accepts, so resample before writing/muxing.
rec_trace_faded = apply_edge_fade(rec_trace, fade_samples=50)

native_sr_int = int(round(playback_sr))
g = gcd(standard_sr, native_sr_int)
up, down = standard_sr // g, native_sr_int // g
rec_trace_resampled = resample_poly(rec_trace_faded, up, down)

wavfile.write("recovered_whalesong_resampled.wav", standard_sr, to_int16(rec_trace_resampled))
print(f"Saved recovered_whalesong_resampled.wav at {standard_sr} Hz "
      f"(native rate was {playback_sr} Hz)")

In [ ]:
# --- Save the recovered-only animation as a silent video, then mux in the recovered audio ---
anim_rec_only.save(
    "recovered_only_spatial_reconstruction_silent.mp4",
    writer=animation.FFMpegWriter(fps=fps, codec="libx264", bitrate=2000),
)

subprocess.run([
    "ffmpeg", "-y",
    "-i", "recovered_only_spatial_reconstruction_silent.mp4",
    "-i", "recovered_whalesong_resampled.wav",
    "-c:v", "copy", "-c:a", "aac", "-shortest",
    "-map", "0:v:0", "-map", "1:a:0",
    "recovered_whalesong_spacetime_reconstruction.mp4",
], check=True)
print("Saved recovered_whalesong_spacetime_reconstruction.mp4")